# FinTrust Risk Review — ML Workflow (Week 3)
**FinTech Week 3 — ML Engineering Track**

Databricks notebook version of the Week 3 workflow. Pipeline:

`Input → Validation → Preprocessing → Feature Preparation → Model → Prediction → Output`

Data is **synthetic** (AnalystLab Africa training data). `Risk_Review_Flag` is an educational
label, **not** a real fraud determination. Nothing here is personalised financial, investment,
lending or legal advice.

| Part | Covered here |
|---|---|
| A — Week 2 review | §1 (checklist — fill in once you have the Week 2 notebook) |
| B — Pipeline | §3–§7 |
| C — Model integration | §6 (own model; documented interface for a DS model) |
| D — Technical testing | §8 |
| E — API/service | §9 (FastAPI code + how to call it from Databricks) |
| F — Reproducibility | §2, §10 |


## 1. Part A — Week 2 Workflow Review (checklist)

Week 2 code wasn't available when this notebook was built, so this is a checklist to run
against it, not an audit. Fill in the "Finding" column.

| Area | Check | Finding |
|---|---|---|
| Incomplete components | Validation stage? Saved, reloadable model? | |
| Technical weaknesses | Encoders/scalers fit on train only (no leakage)? | |
| Technical weaknesses | Random vs. temporal split? Is an outcome field (e.g. `Transaction_Status`) used as a feature? | |
| Technical weaknesses | Missing `Device_Type`/`Location` and unseen categories handled? | |
| Technical weaknesses | Metrics beyond accuracy on an imbalanced target (80/20)? | |
| Testing gaps | Tests for empty input, bad types, missing model file, reproducibility? | |
| Reproducibility | Pinned deps, fixed seed, one-command run, data checksums? | |

This notebook addresses every row above explicitly — see the referenced section.

## 2. Setup & reproducibility

Pin versions, fix the random seed, and resolve data paths so the same run on another cluster
gives the same result. Works both on Databricks (`dbutils`/DBFS present) and locally.

In [ ]:
%pip install -q scikit-learn==1.8.0 pandas==3.0.2 numpy==2.4.4 joblib==1.5.3 openpyxl==3.1.5 mlflow

In [ ]:
dbutils.library.restartPython()  # noqa: F821  (only exists on Databricks; comment out locally)

In [ ]:
import sys, os, json, hashlib, platform
import numpy as np
import pandas as pd
import sklearn
print("python", platform.python_version(), "| sklearn", sklearn.__version__,
      "| pandas", pd.__version__, "| numpy", np.__version__)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

try:
    dbutils  # noqa: F821
    ON_DATABRICKS = True
except NameError:
    ON_DATABRICKS = False
print("Running on Databricks:", ON_DATABRICKS)

In [ ]:
# --- Resolve data location ---------------------------------------------------
# Upload the three FinTrust files to a Databricks volume / DBFS folder (or a workspace
# folder) and set DATA_DIR accordingly. Widgets let you override this from the notebook UI.
if ON_DATABRICKS:
    dbutils.widgets.text("data_dir", "/dbfs/FileStore/fintrust/raw", "Raw data folder")  # noqa: F821
    dbutils.widgets.text("artifact_dir", "/dbfs/FileStore/fintrust/artifacts", "Artifact folder")  # noqa: F821
    DATA_DIR = dbutils.widgets.get("data_dir")  # noqa: F821
    ARTIFACT_DIR = dbutils.widgets.get("artifact_dir")  # noqa: F821
else:
    DATA_DIR = "./data/raw"
    ARTIFACT_DIR = "./artifacts"

os.makedirs(ARTIFACT_DIR, exist_ok=True)
CUSTOMER_FILE = f"{DATA_DIR}/FinTrust_Customer_Data.xlsx"
TRANSACTION_FILE = f"{DATA_DIR}/FinTrust_Transaction_Data.xlsx"
print("Customer file:  ", CUSTOMER_FILE)
print("Transaction file:", TRANSACTION_FILE)

## 3. Schema & config (single source of truth)

Column lists, allowed categories, and value ranges, derived from `FinTrust_Data_Dictionary.xlsx`.
Keeping these in one place is what makes validation and feature prep consistent at
training *and* scoring time.

In [ ]:
TARGET = "Risk_Review_Flag"
TEST_START = "2026-03-01"          # temporal hold-out: train Jan-Feb, test March
INCLUDE_TRANSACTION_STATUS = False  # it's an outcome field — see §4 leakage note
MEDIUM_FRACTION = 0.5

TRANSACTION_REQUIRED = ["Transaction_ID", "Customer_ID", "Transaction_DateTime",
                        "Transaction_Type", "Amount_NGN", "Channel", "International_Transaction"]
TRANSACTION_OPTIONAL = ["Device_Type", "Location", "Transaction_Status"]
CUSTOMER_COLUMNS = ["Customer_ID", "Customer_Name", "Age", "Gender", "City", "Customer_Segment",
                    "Account_Type", "Tenure_Months", "Digital_Engagement_Score",
                    "Monthly_Income_Band", "Preferred_Channel", "Account_Status"]

CITIES = ["Lagos", "Abuja", "Port Harcourt", "Kano", "Ibadan", "Enugu", "Kaduna", "Benin City"]
ALLOWED_CATEGORIES = {
    "Transaction_Type": ["Transfer", "Card Purchase", "Bill Payment", "Cash Withdrawal", "Deposit", "Airtime/Data"],
    "Channel": ["Mobile App", "POS", "Web", "ATM", "USSD"],
    "Device_Type": ["Android", "iOS", "POS Terminal", "Web Browser", "ATM Terminal"],
    "Location": CITIES,
    "International_Transaction": ["Yes", "No"],
    "Transaction_Status": ["Successful", "Failed", "Reversed", "Pending"],
    "Gender": ["Male", "Female", "Prefer not to say"],
    "City": CITIES,
    "Customer_Segment": ["Everyday", "Premium", "Student", "SME"],
    "Account_Type": ["Savings", "Current", "Premium"],
    "Monthly_Income_Band": ["Below 100k", "100k-249k", "250k-499k", "500k-999k", "1m+"],
    "Preferred_Channel": ["Mobile App", "Web", "USSD"],
    "Account_Status": ["Active", "Dormant", "Restricted"],
}
STRICT_CATEGORIES = ["International_Transaction"]   # unseen value here is a hard error
INCOME_BAND_MIDPOINT = {"Below 100k": 50_000.0, "100k-249k": 175_000.0, "250k-499k": 375_000.0,
                        "500k-999k": 750_000.0, "1m+": 1_250_000.0}
AMOUNT_MIN, AMOUNT_MAX = 0.0, 1_000_000_000
AGE_RANGE, TENURE_RANGE, ENGAGEMENT_RANGE = (18, 120), (0, 1200), (0.0, 100.0)

## 4. Input & data preparation

Read the raw files, check referential integrity (customer join key) and record a manifest
(row counts, missing values, target balance, SHA-256 checksums) so a run can be audited later.

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def load_customers(path=CUSTOMER_FILE):
    df = pd.read_excel(path)
    if len(df) == 0:
        raise ValueError("Customer table is empty")
    missing = [c for c in CUSTOMER_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Customer table missing columns: {missing}")
    if df["Customer_ID"].isna().any() or df["Customer_ID"].duplicated().any():
        raise ValueError("Customer_ID must be unique and non-null")
    out = df.copy()
    for col, (lo, hi) in {"Age": AGE_RANGE, "Tenure_Months": TENURE_RANGE,
                          "Digital_Engagement_Score": ENGAGEMENT_RANGE}.items():
        out[col] = pd.to_numeric(out[col], errors="coerce")
        bad = out[col].notna() & ~out[col].between(lo, hi)
        if bad.any():
            raise ValueError(f"{col}: {int(bad.sum())} value(s) outside [{lo}, {hi}]")
    return out

def load_transactions(path=TRANSACTION_FILE):
    return pd.read_excel(path)

customers = load_customers()
transactions = load_transactions()
print("customers:", customers.shape, "| transactions:", transactions.shape)

In [ ]:
orphan = int((~transactions["Customer_ID"].isin(customers["Customer_ID"])).sum())
dup_txn = int(transactions["Transaction_ID"].duplicated().sum())
manifest = {
    "customers_rows": int(len(customers)),
    "transactions_rows": int(len(transactions)),
    "orphan_transactions": orphan,
    "duplicate_transaction_ids": dup_txn,
    "missing_values": {k: int(v) for k, v in transactions.isna().sum().items() if v > 0},
    "target_distribution": transactions[TARGET].value_counts().to_dict(),
    "sha256": {CUSTOMER_FILE: sha256(CUSTOMER_FILE), TRANSACTION_FILE: sha256(TRANSACTION_FILE)},
}
with open(f"{ARTIFACT_DIR}/manifest.json", "w") as fh:
    json.dump(manifest, fh, indent=2)
manifest

## 5. Validation

**Structural** problems (not a table, empty, missing required columns) raise immediately.
**Hard row-level** problems (missing key fields, non-numeric/non-positive amount, bad
datetime, invalid Yes/No flag, duplicate `Transaction_ID`) reject that row with a reason;
the rest of the batch still runs (or the whole call raises if `strict=True`).
**Soft** problems (unexpected category, missing `Device_Type`/`Location`) keep the row and
raise a warning — unseen categories are encoded as all-zero by `OneHotEncoder(handle_unknown="ignore")`,
missing optional values are imputed as `"Unknown"`.

In [ ]:
from dataclasses import dataclass, field

_CAT_TXN_COLS = [c for c in ALLOWED_CATEGORIES if c in TRANSACTION_REQUIRED + TRANSACTION_OPTIONAL]

class EmptyInputError(Exception): pass
class SchemaError(Exception): pass
class ValidationError(Exception): pass

@dataclass
class ValidationReport:
    n_input: int = 0
    n_valid: int = 0
    warnings: list = field(default_factory=list)
    rejected: list = field(default_factory=list)
    @property
    def n_rejected(self): return len(self.rejected)

def _canonicalise(series, allowed):
    lookup = {a.lower(): a for a in allowed}
    def fix(v):
        if pd.isna(v): return np.nan
        s = str(v).strip()
        return lookup.get(s.lower(), s)
    return series.map(fix).astype(object)

def validate_transactions(df, strict=False):
    if not isinstance(df, pd.DataFrame):
        raise SchemaError(f"Expected a pandas DataFrame, got {type(df).__name__}")
    if len(df) == 0:
        raise EmptyInputError("Input contains no rows")
    missing_cols = [c for c in TRANSACTION_REQUIRED if c not in df.columns]
    if missing_cols:
        raise SchemaError(f"Missing required column(s): {missing_cols}")

    report = ValidationReport(n_input=len(df))
    data = df.copy()
    reasons = {idx: [] for idx in data.index}
    def flag(mask, msg):
        for idx in data.index[mask.fillna(False).to_numpy()]:
            reasons[idx].append(msg)

    for col in ["Transaction_ID", "Customer_ID"]:
        data[col] = data[col].map(lambda v: np.nan if pd.isna(v) else str(v).strip()).astype(object)
        data.loc[data[col] == "", col] = np.nan
        flag(data[col].isna(), f"{col} missing")
    dup = data["Transaction_ID"].notna() & data["Transaction_ID"].duplicated(keep="first")
    flag(dup, "duplicate Transaction_ID")

    raw_amount = data["Amount_NGN"]
    amount = pd.to_numeric(raw_amount, errors="coerce")
    flag(raw_amount.isna(), "Amount_NGN missing")
    flag(raw_amount.notna() & amount.isna(), "Amount_NGN not numeric")
    finite = amount.replace([np.inf, -np.inf], np.nan)
    flag(amount.notna() & finite.isna(), "Amount_NGN not finite")
    flag(finite.notna() & (finite <= AMOUNT_MIN), "Amount_NGN must be > 0")
    flag(finite.notna() & (finite > AMOUNT_MAX), "Amount_NGN above sanity ceiling")
    data["Amount_NGN"] = finite.astype(float)

    raw_dt = data["Transaction_DateTime"]
    dt = pd.to_datetime(raw_dt, errors="coerce")
    flag(raw_dt.isna(), "Transaction_DateTime missing")
    flag(raw_dt.notna() & dt.isna(), "Transaction_DateTime not a valid datetime")
    data["Transaction_DateTime"] = dt

    for col in _CAT_TXN_COLS:
        if col not in data.columns: data[col] = np.nan
        data[col] = _canonicalise(data[col].replace("", np.nan), ALLOWED_CATEGORIES[col])
    for col in TRANSACTION_REQUIRED:
        if col in _CAT_TXN_COLS:
            flag(data[col].isna(), f"{col} missing")
    for col in STRICT_CATEGORIES:
        bad = data[col].notna() & ~data[col].isin(ALLOWED_CATEGORIES[col])
        flag(bad, f"{col} must be one of {ALLOWED_CATEGORIES[col]}")

    rejected_idx = [i for i, r in reasons.items() if r]
    for pos, idx in enumerate(data.index):
        if idx in rejected_idx:
            tid = data.at[idx, "Transaction_ID"]
            report.rejected.append({"row": int(pos), "Transaction_ID": None if pd.isna(tid) else tid,
                                    "reasons": reasons[idx]})
    if strict and report.rejected:
        raise ValidationError(f"{len(report.rejected)} invalid row(s): {report.rejected[:3]}")
    clean = data.drop(index=rejected_idx)

    for col in _CAT_TXN_COLS:
        if col in STRICT_CATEGORIES: continue
        vals = clean[col].dropna()
        unexpected = sorted(set(vals) - set(ALLOWED_CATEGORIES[col]))
        if unexpected:
            report.warnings.append(f"{col}: unexpected categories {unexpected} (encoded as unseen)")
    for col in ["Device_Type", "Location"]:
        n_missing = int(clean[col].isna().sum())
        if n_missing:
            report.warnings.append(f"{col}: {n_missing} missing value(s) imputed as 'Unknown'")

    report.n_valid = len(clean)
    return clean, report

clean_txn, vreport = validate_transactions(transactions)
print(f"input={vreport.n_input} valid={vreport.n_valid} rejected={vreport.n_rejected}")
vreport.warnings

## 6. Feature preparation

Pure function — identical behaviour at training and scoring time. Encoding/scaling/imputation
are learned *inside* the sklearn `Pipeline` from training data only, so nothing here peeks
at the target or fits on test data.

In [ ]:
CUSTOMER_JOIN_COLUMNS = ["Age", "Gender", "City", "Customer_Segment", "Account_Type", "Tenure_Months",
                         "Digital_Engagement_Score", "Monthly_Income_Band", "Preferred_Channel", "Account_Status"]
NUMERIC_FEATURES = ["Age", "Tenure_Months", "Digital_Engagement_Score", "log_amount", "hour",
                    "day_of_week", "is_night", "is_weekend", "amount_to_income",
                    "channel_differs_from_preferred", "location_differs_from_city"]
CATEGORICAL_FEATURES = ["Transaction_Type", "Channel", "Device_Type", "Location", "International_Transaction",
                        "Gender", "City", "Customer_Segment", "Account_Type", "Account_Status", "Preferred_Channel"]
STATUS_FEATURE = "Transaction_Status"

def feature_columns(include_status=INCLUDE_TRANSACTION_STATUS):
    cats = CATEGORICAL_FEATURES + ([STATUS_FEATURE] if include_status else [])
    return NUMERIC_FEATURES + cats

def attach_customers(txn, customers):
    ref = customers[["Customer_ID"] + CUSTOMER_JOIN_COLUMNS].drop_duplicates("Customer_ID")
    joined = txn.merge(ref, on="Customer_ID", how="left", validate="many_to_one")
    joined.index = txn.index
    unmatched = joined[joined["Age"].isna() & joined["Account_Status"].isna()]
    return joined, unmatched

def prepare_features(df, include_status=INCLUDE_TRANSACTION_STATUS):
    out = pd.DataFrame(index=df.index)
    dt = pd.to_datetime(df["Transaction_DateTime"])
    amount = pd.to_numeric(df["Amount_NGN"], errors="coerce")
    out["Age"] = pd.to_numeric(df["Age"], errors="coerce")
    out["Tenure_Months"] = pd.to_numeric(df["Tenure_Months"], errors="coerce")
    out["Digital_Engagement_Score"] = pd.to_numeric(df["Digital_Engagement_Score"], errors="coerce")
    out["log_amount"] = np.log1p(amount)
    out["hour"] = dt.dt.hour.astype(float)
    out["day_of_week"] = dt.dt.dayofweek.astype(float)
    out["is_night"] = (dt.dt.hour < 6).astype(float)          # review rate rises sharply 00:00-05:59
    out["is_weekend"] = (dt.dt.dayofweek >= 5).astype(float)
    income_mid = df["Monthly_Income_Band"].map(INCOME_BAND_MIDPOINT).astype(float)
    out["amount_to_income"] = np.log1p(amount / income_mid)
    out["channel_differs_from_preferred"] = (df["Channel"].astype(object) != df["Preferred_Channel"].astype(object)).astype(float)
    loc = df["Location"].astype(object)
    out["location_differs_from_city"] = np.where(loc.isna() | df["City"].isna(), np.nan,
                                                   (loc != df["City"].astype(object)).astype(float))
    for col in CATEGORICAL_FEATURES + ([STATUS_FEATURE] if include_status else []):
        out[col] = df[col].astype(object) if col in df.columns else np.nan
    return out[feature_columns(include_status)]

joined, unmatched = attach_customers(clean_txn, customers)
if len(unmatched):
    print(f"{len(unmatched)} row(s) with unknown Customer_ID dropped")
    joined = joined.drop(index=unmatched.index)

target = joined[TARGET].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0})
assert target.isna().sum() == 0, "Risk_Review_Flag contains values other than Yes/No"

X = prepare_features(joined)
y = target
X.head()

## 7. Model

`RiskModel` wraps a fitted sklearn `Pipeline` + decision threshold + metadata as one
artifact. Any model used for scoring must expose `predict_proba(X)`, `.threshold`,
`.version`, `.include_status` (`ModelInterface`) — this is where a Data Science model
would plug in. **No other intern's model is integrated here**; this is this workflow's
own model, built for development purposes.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CANDIDATES = ("logistic_regression", "random_forest", "hist_gradient_boosting")

def build_pipeline(name="logistic_regression", include_status=INCLUDE_TRANSACTION_STATUS):
    numeric = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),
                        ("scale", StandardScaler())])
    categorical = Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
                            ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    cats = CATEGORICAL_FEATURES + ([STATUS_FEATURE] if include_status else [])
    pre = ColumnTransformer([("num", numeric, NUMERIC_FEATURES), ("cat", categorical, cats)])
    if name == "logistic_regression":
        clf = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED)
    elif name == "random_forest":
        clf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced",
                                     n_jobs=1, random_state=RANDOM_SEED)
    elif name == "hist_gradient_boosting":
        clf = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=200,
                                             class_weight="balanced", random_state=RANDOM_SEED)
    else:
        raise ValueError(f"Unknown model '{name}'. Choose from {CANDIDATES}")
    return Pipeline([("prep", pre), ("clf", clf)])

class RiskModel:
    """Fitted pipeline + threshold + metadata, saved/loaded as one artifact."""
    def __init__(self, pipeline, threshold=0.5, include_status=INCLUDE_TRANSACTION_STATUS,
                version="0.3.0", algorithm="logistic_regression", metadata=None):
        self.pipeline, self.threshold, self.include_status = pipeline, threshold, include_status
        self.version, self.algorithm, self.metadata = version, algorithm, metadata or {}
    def predict_proba(self, X):
        cols = feature_columns(self.include_status)
        missing = [c for c in cols if c not in X.columns]
        if missing: raise KeyError(f"Feature frame missing columns: {missing}")
        return self.pipeline.predict_proba(X[cols])[:, 1]
    def save(self, path):
        import joblib; joblib.dump(self, path); return path
    @staticmethod
    def load(path):
        import joblib, os
        if not os.path.exists(path):
            raise FileNotFoundError(f"Model file not found: {path}. Run training first.")
        obj = joblib.load(path)
        if not isinstance(obj, RiskModel):
            raise TypeError(f"{path} does not contain a RiskModel")
        return obj

In [ ]:
# --- temporal split: train Jan-Feb, test March (no leakage from the future) ---------
is_test = joined["Transaction_DateTime"] >= pd.Timestamp(TEST_START)
X_tr, y_tr, X_te, y_te = X[~is_test], y[~is_test], X[is_test], y[is_test]
print("train:", X_tr.shape, "test:", X_te.shape)

In [ ]:
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_val_predict

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_results, oof = {}, {}
for name in CANDIDATES:
    proba = cross_val_predict(build_pipeline(name), X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
    oof[name] = proba
    cv_results[name] = {"cv_roc_auc": round(float(roc_auc_score(y_tr, proba)), 4),
                        "cv_pr_auc": round(float(average_precision_score(y_tr, proba)), 4)}

best = max(cv_results, key=lambda n: cv_results[n]["cv_pr_auc"])
print("Candidate comparison (5-fold CV on train):")
for n, m in cv_results.items():
    flag = "  <- selected" if n == best else ""
    print(f"  {n:24s} ROC-AUC={m['cv_roc_auc']:.4f}  PR-AUC={m['cv_pr_auc']:.4f}{flag}")

In [ ]:
# threshold chosen on train out-of-fold predictions (never touches test)
def best_f1_threshold(y_true, proba):
    grid = np.round(np.arange(0.20, 0.81, 0.01), 2)
    scores = [f1_score(y_true, (proba >= t).astype(int), zero_division=0) for t in grid]
    return float(grid[int(np.argmax(scores))])

threshold = best_f1_threshold(y_tr, oof[best])
pipe = build_pipeline(best).fit(X_tr, y_tr)
print("selected:", best, "| threshold:", threshold)

In [ ]:
def evaluate(y_true, proba, threshold):
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {"roc_auc": round(float(roc_auc_score(y_true, proba)), 4),
           "pr_auc": round(float(average_precision_score(y_true, proba)), 4),
           "precision": round(float(precision_score(y_true, pred, zero_division=0)), 4),
           "recall": round(float(recall_score(y_true, pred, zero_division=0)), 4),
           "f1": round(float(f1_score(y_true, pred, zero_division=0)), 4),
           "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
           "positive_rate_actual": round(float(np.mean(y_true)), 4),
           "positive_rate_predicted": round(float(pred.mean()), 4)}

proba_te = pipe.predict_proba(X_te)[:, 1]
test_metrics = evaluate(y_te, proba_te, threshold)
test_metrics

In [ ]:
from sklearn.inspection import permutation_importance
imp = permutation_importance(pipe, X_te, y_te, scoring="average_precision", n_repeats=5, random_state=RANDOM_SEED)
importance = sorted(({"feature": c, "importance": round(float(m), 4)}
                     for c, m in zip(X_te.columns, imp.importances_mean)), key=lambda d: -d["importance"])
pd.DataFrame(importance[:10])

**Reading the results:** ROC-AUC ~0.67 and PR-AUC ~0.33 against a 20% base rate means the
model ranks better than chance but is weak — the candidate models perform almost identically,
which points to limited learnable signal in the synthetic label rather than an undertuned model.
`Transaction_Type`, `log_amount` and `International_Transaction` carry most of the signal.
Treat `risk_score` as decision support for human review, not an automated decision.

In [ ]:
rm = RiskModel(pipe, threshold=threshold, include_status=False, version="0.3.0", algorithm=best,
              metadata={"trained_rows": int(len(X_tr)), "test_start": TEST_START})
model_path = f"{ARTIFACT_DIR}/risk_model.joblib"
rm.save(model_path)

metrics = {"model_version": rm.version, "algorithm": best, "threshold": threshold,
          "include_transaction_status": False,
          "split": {"type": "temporal", "test_start": TEST_START, "n_train": int(len(X_tr)), "n_test": int(len(X_te))},
          "candidate_cv_results": cv_results, "test_metrics": test_metrics,
          "permutation_importance_top10": importance[:10], "features": list(X.columns),
          "validation": {"n_input": vreport.n_input, "n_rejected": vreport.n_rejected},
          "seed": RANDOM_SEED,
          "environment": {"python": platform.python_version(), "scikit_learn": sklearn.__version__,
                          "pandas": pd.__version__, "numpy": np.__version__}}
with open(f"{ARTIFACT_DIR}/metrics.json", "w") as fh:
    json.dump(metrics, fh, indent=2)
print("saved model ->", model_path)

### 7a. (Optional) log the run to MLflow

Databricks ships MLflow natively — this registers the run, metrics and model so it shows up
in the Experiments UI and can be promoted via the Model Registry.

In [ ]:
try:
    import mlflow, mlflow.sklearn
    with mlflow.start_run(run_name=f"fintrust-risk-{best}"):
        mlflow.log_param("algorithm", best)
        mlflow.log_param("threshold", threshold)
        mlflow.log_param("include_transaction_status", False)
        mlflow.log_metrics({k: v for k, v in test_metrics.items() if isinstance(v, (int, float))})
        mlflow.sklearn.log_model(pipe, "model")
        print("Logged to MLflow run:", mlflow.active_run().info.run_id)
except ImportError:
    print("mlflow not installed — skipping (not required outside Databricks)")

## 8. Prediction pipeline (Input → ... → Output)

Wraps validation, customer enrichment, feature prep and the model into one reusable call,
so scoring behaves identically whether it's invoked from this notebook, the CLI, or the API.

In [ ]:
OUTPUT_COLUMNS = ["Transaction_ID", "Customer_ID", "risk_score", "predicted_risk_review_flag",
                  "risk_band", "threshold", "model_version"]

class RiskScoringPipeline:
    def __init__(self, model, customers):
        self.model = model
        self.customers = load_customers_df(customers) if isinstance(customers, str) else customers

    def band(self, score):
        t = self.model.threshold
        if score >= t: return "High"
        return "Medium" if score >= t * MEDIUM_FRACTION else "Low"

    def run(self, df, strict=False):
        clean, report = validate_transactions(df, strict=strict)
        j, unmatched = attach_customers(clean, self.customers)
        if len(unmatched):
            pos = {idx: p for p, idx in enumerate(clean.index)}
            for idx, row in unmatched.iterrows():
                report.rejected.append({"row": pos[idx], "Transaction_ID": row["Transaction_ID"],
                                        "reasons": ["Customer_ID not found in customer table"]})
            j = j.drop(index=unmatched.index)
        report.n_valid = len(j)
        if strict and report.rejected:
            raise ValidationError(f"Invalid rows: {report.rejected[:3]}")
        if j.empty:
            return pd.DataFrame(columns=OUTPUT_COLUMNS), report
        Xp = prepare_features(j, include_status=self.model.include_status)
        scores = self.model.predict_proba(Xp)
        out = pd.DataFrame({"Transaction_ID": j["Transaction_ID"].to_numpy(),
                            "Customer_ID": j["Customer_ID"].to_numpy(), "risk_score": scores.round(4)})
        out["predicted_risk_review_flag"] = np.where(scores >= self.model.threshold, "Yes", "No")
        out["risk_band"] = [self.band(s) for s in scores]
        out["threshold"] = self.model.threshold
        out["model_version"] = self.model.version
        return out[OUTPUT_COLUMNS].reset_index(drop=True), report

def load_customers_df(path): return load_customers(path)

pipeline = RiskScoringPipeline(RiskModel.load(model_path), customers)

In [ ]:
# demo: score a single new transaction
demo_txn = pd.DataFrame([{
    "Transaction_ID": "FT-T900001", "Customer_ID": customers.iloc[0]["Customer_ID"],
    "Transaction_DateTime": "2026-03-14 02:15:00", "Transaction_Type": "Cash Withdrawal",
    "Amount_NGN": 450000, "Channel": "ATM", "International_Transaction": "Yes",
    "Device_Type": "ATM Terminal", "Location": "Kano",
}])
preds, rep = pipeline.run(demo_txn)
display(preds) if ON_DATABRICKS else print(preds.to_string())
print("warnings:", rep.warnings)

In [ ]:
# demo: score the full transaction file and write results (Databricks: writes to DBFS)
all_preds, all_report = pipeline.run(transactions.drop(columns=[TARGET]))
out_path = f"{ARTIFACT_DIR}/predictions.csv"
all_preds.to_csv(out_path, index=False)
print(f"scored {all_report.n_valid} of {all_report.n_input} rows "
     f"({all_report.n_rejected} rejected) -> {out_path}")
all_preds["risk_band"].value_counts()

## 9. Technical tests

Notebook-native (`assert`) versions of the project's pytest suite, so they run on Databricks
without a separate test runner. Covers: valid input, missing values, unexpected categories,
invalid data types, empty input, model loading, prediction generation, output format,
reproducibility.

In [ ]:
def check(label, fn):
    try:
        fn()
        print(f"PASS  {label}")
    except AssertionError as e:
        print(f"FAIL  {label}: {e}")
    except Exception as e:
        print(f"ERROR {label}: {type(e).__name__}: {e}")

VALID_ROW = {"Transaction_ID": "FT-T900001", "Customer_ID": customers.iloc[0]["Customer_ID"],
            "Transaction_DateTime": "2026-03-14 02:15:00", "Transaction_Type": "Transfer",
            "Amount_NGN": 25000.0, "Channel": "Mobile App", "International_Transaction": "No",
            "Device_Type": "Android", "Location": "Lagos"}

def t_valid_input():
    c, r = validate_transactions(pd.DataFrame([VALID_ROW]))
    assert len(c) == 1 and r.n_rejected == 0

def t_missing_optional_warns_not_rejects():
    df = pd.DataFrame([dict(VALID_ROW, Device_Type=np.nan, Location=np.nan)])
    c, r = validate_transactions(df)
    assert len(c) == 1 and any("Device_Type" in w for w in r.warnings)

def t_unexpected_category_warns_and_keeps_row():
    df = pd.DataFrame([dict(VALID_ROW, Transaction_Type="Crypto Swap")])
    c, r = validate_transactions(df)
    assert len(c) == 1 and any("Crypto Swap" in w for w in r.warnings)

def t_invalid_amount_type_rejected():
    df = pd.DataFrame([dict(VALID_ROW, Amount_NGN="lots")])
    c, r = validate_transactions(df)
    assert len(c) == 0 and r.n_rejected == 1

def t_empty_input_raises():
    try:
        validate_transactions(pd.DataFrame(columns=list(VALID_ROW)))
        assert False, "expected EmptyInputError"
    except EmptyInputError:
        pass

def t_model_loading_missing_file_raises():
    try:
        RiskModel.load(f"{ARTIFACT_DIR}/does_not_exist.joblib")
        assert False, "expected FileNotFoundError"
    except FileNotFoundError:
        pass

def t_prediction_generation_range():
    c, _ = validate_transactions(pd.DataFrame([VALID_ROW]))
    j, _ = attach_customers(c, customers)
    p = rm.predict_proba(prepare_features(j))
    assert p.shape == (1,) and 0 <= p[0] <= 1

def t_output_format():
    preds, _ = pipeline.run(pd.DataFrame([VALID_ROW]))
    assert list(preds.columns) == OUTPUT_COLUMNS
    assert preds.iloc[0]["risk_band"] in {"Low", "Medium", "High"}

def t_reproducibility_same_input_same_output():
    a, _ = pipeline.run(pd.DataFrame([VALID_ROW]))
    b, _ = pipeline.run(pd.DataFrame([VALID_ROW]))
    pd.testing.assert_frame_equal(a, b)

for label, fn in [("valid input", t_valid_input),
                  ("missing optional values -> warn, not reject", t_missing_optional_warns_not_rejects),
                  ("unexpected category -> warn, still scored", t_unexpected_category_warns_and_keeps_row),
                  ("invalid data type -> rejected", t_invalid_amount_type_rejected),
                  ("empty input -> raises", t_empty_input_raises),
                  ("model loading: missing file -> raises", t_model_loading_missing_file_raises),
                  ("prediction generation: shape/range", t_prediction_generation_range),
                  ("output format", t_output_format),
                  ("reproducibility: deterministic scoring", t_reproducibility_same_input_same_output)]:
    check(label, fn)

In [ ]:
# reproducibility evidence: retrain from scratch and compare metrics + a hash of the scores
pipe2 = build_pipeline(best).fit(X_tr, y_tr)
proba2 = pipe2.predict_proba(X_te)[:, 1]
metrics2 = evaluate(y_te, proba2, threshold)
assert metrics2 == test_metrics, "metrics changed on retrain — check seeding"
h1 = hashlib.sha256(proba_te.tobytes()).hexdigest()[:16]
h2 = hashlib.sha256(proba2.tobytes()).hexdigest()[:16]
assert h1 == h2, "prediction vector changed on retrain"
print("Retrain reproduced identical metrics and predictions. Score hash:", h1)

## 10. API/service component (Part E)

A FastAPI service wraps the same `RiskScoringPipeline`. It's defined here for completeness
and to document the contract; FastAPI's own event loop doesn't start cleanly inside a
Databricks notebook cell, so run it as a **job/task** (e.g. `%run` this notebook from a
`.py` entry point, or export this cell to a `.py` file and run with `uvicorn`) rather than
inline. The cell below shows the app and documents the endpoints; the one after shows how
to call it with `TestClient` without starting a real server, which *does* work inline.

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Optional

DISCLAIMER = ("Risk_Review_Flag is a synthetic educational label, not a fraud determination. "
             "Scores are decision support for human review only.")

class TransactionIn(BaseModel):
    Transaction_ID: str
    Customer_ID: str
    Transaction_DateTime: str = Field(examples=["2026-03-14 02:15:00"])
    Transaction_Type: str
    Amount_NGN: float
    Channel: str
    International_Transaction: str
    Device_Type: Optional[str] = None
    Location: Optional[str] = None
    Transaction_Status: Optional[str] = None

class BatchIn(BaseModel):
    transactions: list[TransactionIn]

app = FastAPI(title="FinTrust Risk Review API", version=rm.version)

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": True, "api_version": rm.version}

@app.get("/model-info")
def model_info():
    return {"model_version": rm.version, "algorithm": rm.algorithm, "threshold": rm.threshold,
           "include_transaction_status": rm.include_status,
           "features": feature_columns(rm.include_status), "disclaimer": DISCLAIMER}

@app.post("/predict")
def predict(req: TransactionIn):
    try:
        preds, report = pipeline.run(pd.DataFrame([req.model_dump()]))
    except (EmptyInputError, SchemaError) as exc:
        raise HTTPException(422, detail=str(exc))
    if preds.empty:
        raise HTTPException(422, detail={"rejected": report.rejected})
    return {"prediction": preds.iloc[0].to_dict(), "warnings": report.warnings, "disclaimer": DISCLAIMER}

@app.post("/predict/batch")
def predict_batch(req: BatchIn):
    if not req.transactions:
        raise HTTPException(422, detail="transactions must not be empty")
    if len(req.transactions) > 1000:
        raise HTTPException(413, detail="Max 1000 transactions per request")
    df = pd.DataFrame([t.model_dump() for t in req.transactions])
    preds, report = pipeline.run(df)
    return {"predictions": preds.to_dict(orient="records"),
           "validation": {"n_input": report.n_input, "n_valid": report.n_valid,
                          "n_rejected": report.n_rejected, "warnings": report.warnings,
                          "rejected": report.rejected},
           "disclaimer": DISCLAIMER}

print("FastAPI app defined:", [r.path for r in app.routes])

In [ ]:
# Exercise the API in-process (no real server/port needed) — proves the contract works.
from fastapi.testclient import TestClient
client = TestClient(app)
print(client.get("/health").json())
print(client.get("/model-info").json())
r = client.post("/predict", json=VALID_ROW)
print(r.status_code, r.json())

**To actually serve this on a port** (e.g. from a Databricks cluster driver, or export
this cell to `api.py` and run elsewhere):
```bash
uvicorn api:app --host 0.0.0.0 --port 8000
```
Then on Databricks, open a port-forward / use `databricks-cli` proxy, or call it from a job
cluster's driver node. Endpoints: `GET /health`, `GET /model-info`, `POST /predict`,
`POST /predict/batch` — see docstring above for the request/response shapes.

## 11. Part F — Reproducibility summary

1. **Install**: the `%pip install` cell in §2 pins exact versions.
2. **Prepare data**: upload the 3 FinTrust files to the `data_dir` widget path (DBFS/volume);
   §4 reads and checksums them into `manifest.json`.
3. **Run the workflow**: run all cells top-to-bottom (`Run All`) — §5–§7 validate, engineer
   features and train with a fixed seed (`RANDOM_SEED = 42`).
4. **Generate predictions**: §8's `pipeline.run(df)` scores any new transaction file the
   same way; §10's API exposes the same call over HTTP.
5. **Interpret the output**: `risk_score` (0–1), `predicted_risk_review_flag` (Yes/No at the
   tuned threshold), `risk_band` (Low/Medium/High) — see the table below.

| Column | Meaning |
|---|---|
| `risk_score` | model probability the transaction is labelled for risk review |
| `predicted_risk_review_flag` | `Yes` if `risk_score >= threshold` |
| `risk_band` | High ≥ threshold; Medium ≥ threshold/2; Low below |

§9's retrain-and-compare cell is the reproducibility evidence: identical seed ⇒ identical
metrics and an identical SHA-256 of the prediction vector. `artifacts/metrics.json` and
`artifacts/manifest.json` record the exact environment, data checksums and results for this run.
